# Data Collection
Combines O\*NET, BLS OEWS (wages/employment), and BLS Employment Projections into a single master dataframe keyed by SOC code. Also computes an automation risk score from O\*NET task data.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')

## 1. O\*NET master

In [2]:
onet = pd.read_parquet(PROCESSED / 'onet_master.parquet')
# 7-char SOC prefix for joining with BLS (e.g. '15-1252' from '15-1252.00')
onet['soc_7'] = onet['soc_code'].str[:7]
print(f"O*NET: {len(onet):,} occupations, {onet.shape[1]} columns")

O*NET: 1,016 occupations, 446 columns


## 2. BLS OEWS — employment & wages

In [3]:
oews_raw = pd.read_excel(RAW / 'bls_oews.xlsx', sheet_name=0)

# Keep national cross-industry detailed occupations only
oews = oews_raw[
    (oews_raw['I_GROUP'] == 'cross-industry') &
    (oews_raw['O_GROUP'] == 'detailed')
].copy()

oews = oews[[
    'OCC_CODE', 'OCC_TITLE',
    'TOT_EMP',              # total employment
    'A_MEAN',               # annual mean wage
    'A_MEDIAN',             # annual median wage
    'A_PCT10', 'A_PCT25', 'A_PCT75', 'A_PCT90',  # wage distribution
    'JOBS_1000',            # jobs per 1,000 employed (concentration)
]].rename(columns={
    'OCC_CODE':  'soc_7',
    'OCC_TITLE': 'bls_title',
    'TOT_EMP':   'employment',
    'A_MEAN':    'annual_mean_wage',
    'A_MEDIAN':  'annual_median_wage',
    'A_PCT10':   'wage_pct10',
    'A_PCT25':   'wage_pct25',
    'A_PCT75':   'wage_pct75',
    'A_PCT90':   'wage_pct90',
    'JOBS_1000': 'jobs_per_1000',
})

# Numeric coercion (BLS uses '*' and '#' for suppressed/non-applicable values)
wage_cols = ['employment','annual_mean_wage','annual_median_wage',
             'wage_pct10','wage_pct25','wage_pct75','wage_pct90','jobs_per_1000']
for c in wage_cols:
    oews[c] = pd.to_numeric(oews[c], errors='coerce')

print(f"OEWS: {len(oews):,} detailed occupations")
oews.head(3)

OEWS: 830 detailed occupations


,soc_7,bls_title,employment,annual_mean_wage,annual_median_wage,wage_pct10,wage_pct25,wage_pct75,wage_pct90,jobs_per_1000
4,11-1011,Chief Executives,204350,269630.0,213990.0,75700.0,129540.0,356200.0,507730.0,NaN
6,11-1021,General and Operations Managers,3503020,134940.0,105770.0,50090.0,72320.0,167280.0,253390.0,NaN
9,11-2011,Advertising and Promotions Managers,21470,154280.0,133660.0,63300.0,91370.0,201050.0,286240.0,NaN


## 3. BLS Employment Projections — growth & openings

In [4]:
proj_raw = pd.read_excel(RAW / 'bls_projections.xlsx', sheet_name='Table 1.2', header=1)

proj_raw.columns = [
    'title', 'soc_code_raw', 'occ_type',
    'emp_2024', 'emp_2034',
    'emp_dist_pct_2024', 'emp_dist_pct_2034',
    'emp_change_num', 'emp_change_pct',
    'pct_self_employed',
    'annual_openings',
    'median_wage_2024',
    'entry_education',
    'work_experience',
    'on_job_training',
    'ooh_link'
]

proj = proj_raw[proj_raw['occ_type'] == 'Line item'].copy()
proj['title'] = proj['title'].str.strip()
proj['soc_7'] = proj['soc_code_raw'].astype(str).str.strip()

num_cols = ['emp_change_pct', 'emp_change_num', 'annual_openings',
            'emp_2024', 'emp_2034', 'median_wage_2024']
for c in num_cols:
    proj[c] = pd.to_numeric(proj[c], errors='coerce')

proj = proj[[
    'soc_7', 'emp_2024', 'emp_2034',
    'emp_change_num', 'emp_change_pct',
    'annual_openings', 'median_wage_2024',
    'entry_education', 'work_experience', 'on_job_training'
]]

print(f"Projections: {len(proj):,} detailed occupations")
proj.head(3)

Projections: 832 detailed occupations


,soc_7,emp_2024,emp_2034,emp_change_num,emp_change_pct,annual_openings,median_wage_2024,entry_education,work_experience,on_job_training
3,11-1011,309.4,322.7,13.3,4.3,22.2,206420.0,Bachelor's degree,5 years or more,None
4,11-1021,3712.9,3876.8,164.0,4.4,308.7,102950.0,Bachelor's degree,5 years or more,None
5,11-1031,27.7,28.6,0.9,3.4,2.2,44810.0,Bachelor's degree,Less than 5 years,None


## 4. Automation risk score from O\*NET task data

We derive an automation risk score using the same logic as Frey & Osborne (2013) but with current O\*NET 30.2 data.
Roles with high routine cognitive/physical task intensity and low social/creative requirements score higher.

**High-risk features** (routine / easily automatable):
- Processing Information, Documenting/Recording Information (work activities)
- Repetitive physical work (work context: pace determined by speed of equipment)

**Low-risk features** (protect against automation):
- Social Perceptiveness, Negotiation, Persuasion (skills)
- Originality, Fluency of Ideas (abilities)
- Assisting/Caring for Others, Performing for Others (work activities)

In [5]:
# --- Features that INCREASE automation risk ---
high_risk_features = [
    'work_activity__Processing Information__IM',
    'work_activity__Documenting/Recording Information__IM',
    'work_activity__Performing Administrative Activities__IM',
    'work_activity__Interacting With Computers__IM',
    'work_context__Degree of Automation',
    'work_context__Importance of Repeating Same Tasks',
    'work_context__Pace Determined by Speed of Equipment',
    'skill__Active Learning__IM',       # low -> high risk
]

# --- Features that DECREASE automation risk ---
low_risk_features = [
    'skill__Social Perceptiveness__IM',
    'skill__Negotiation__IM',
    'skill__Persuasion__IM',
    'skill__Service Orientation__IM',
    'skill__Instructing__IM',
    'ability__Originality__IM',
    'ability__Fluency of Ideas__IM',
    'work_activity__Assisting and Caring for Others__IM',
    'work_activity__Performing for or Working Directly with the Public__IM',
    'work_activity__Thinking Creatively__IM',
    'work_activity__Developing and Building Teams__IM',
]

# Keep only columns that exist in onet
high_cols = [c for c in high_risk_features if c in onet.columns]
low_cols  = [c for c in low_risk_features  if c in onet.columns]

print(f"High-risk features found: {len(high_cols)}/{len(high_risk_features)}")
print(f"Low-risk features found:  {len(low_cols)}/{len(low_risk_features)}")

High-risk features found: 7/8
Low-risk features found:  11/11


In [6]:
from sklearn.preprocessing import MinMaxScaler

auto_df = onet[['soc_code', 'title'] + high_cols + low_cols].copy()

# Impute missing with column means
for c in high_cols + low_cols:
    auto_df[c] = auto_df[c].fillna(auto_df[c].mean())

# Normalise all to 0-1
scaler = MinMaxScaler()
auto_df[high_cols + low_cols] = scaler.fit_transform(auto_df[high_cols + low_cols])

# Automation risk = mean(high-risk) - mean(low-risk), rescaled to 0-1
auto_df['raw_score'] = auto_df[high_cols].mean(axis=1) - auto_df[low_cols].mean(axis=1)
auto_df['automation_risk'] = MinMaxScaler().fit_transform(auto_df[['raw_score']]).round(4)

auto_df = auto_df[['soc_code', 'automation_risk']]

print("Automation risk distribution:")
print(auto_df['automation_risk'].describe().round(3))
print("\nHighest automation risk:")
print(auto_df.merge(onet[['soc_code','title']], on='soc_code')
      .sort_values('automation_risk', ascending=False)[['title','automation_risk']].head(8).to_string(index=False))
print("\nLowest automation risk:")
print(auto_df.merge(onet[['soc_code','title']], on='soc_code')
      .sort_values('automation_risk')[['title','automation_risk']].head(8).to_string(index=False))

Automation risk distribution:
count    1016.000
mean        0.527
std         0.162
min         0.000
25%         0.424
50%         0.527
75%         0.632
max         1.000
Name: automation_risk, dtype: float64

Highest automation risk:
                                                                               title  automation_risk
                                                 Chemical Plant and System Operators           1.0000
                                         Court Reporters and Simultaneous Captioners           0.9906
                                                           Medical Transcriptionists           0.9728
Extruding, Forming, Pressing, and Compacting Machine Setters, Operators, and Tenders           0.9612
                                                                   Data Entry Keyers           0.9319
                                                               Histology Technicians           0.9243
                                                

## 5. Join everything into a master dataframe

In [7]:
# Start with O*NET base columns only (skip the 400+ feature cols for this master)
base_cols = [
    'soc_code', 'soc_7', 'title', 'description',
    'job_zone', 'required_education_level',
    'technology_skills', 'hot_technologies',
]
master = onet[base_cols].copy()

# Join automation risk
master = master.merge(auto_df, on='soc_code', how='left')

# Join OEWS
master = master.merge(oews, on='soc_7', how='left')

# Join projections
master = master.merge(proj, on='soc_7', how='left')

print(f"Master: {master.shape}")
print(f"  Occupations with OEWS data:        {master['employment'].notna().sum():,}")
print(f"  Occupations with projection data:  {master['emp_change_pct'].notna().sum():,}")
print(f"  Occupations with automation score: {master['automation_risk'].notna().sum():,}")

Master: (1016, 27)
  Occupations with OEWS data:        961
  Occupations with projection data:  963
  Occupations with automation score: 1,016


In [8]:
# Quick look at key fields
master[['title','employment','annual_median_wage','emp_change_pct','annual_openings','automation_risk']].head(10)

,title,employment,annual_median_wage,emp_change_pct,annual_openings,automation_risk
0,Chief Executives,204350.0,213990.0,4.3,22.2,0.3063
1,Chief Sustainability Officers,204350.0,213990.0,4.3,22.2,0.1955
2,General and Operations Managers,3503020.0,105770.0,4.4,308.7,0.4445
3,Legislators,NaN,NaN,3.4,2.2,0.5270
4,Advertising and Promotions Managers,21470.0,133660.0,-2.2,2.1,0.2977
5,Marketing Managers,395240.0,166790.0,6.6,34.3,0.2400
6,Sales Managers,637080.0,148270.0,4.7,49.0,0.1915
7,Public Relations Managers,74850.0,146910.0,5.0,6.6,0.5270
8,Fundraising Managers,38810.0,125470.0,4.2,3.6,0.3440
9,Administrative Services Managers,263960.0,114130.0,4.6,23.2,0.4791


## 6. Save

In [9]:
master.to_parquet(PROCESSED / 'master.parquet', index=False)
print(f"Saved to data/processed/master.parquet ({master.shape[0]:,} rows x {master.shape[1]} cols)")

Saved to data/processed/master.parquet (1,016 rows x 27 cols)
